In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gzip
import lightgbm

warnings.filterwarnings("ignore")

In [2]:
TRAIN_PATH = "/kaggle/input/competitions/avazu-ctr-prediction/train.gz"
SAMPLE_SIZE = 5000000
RANDOM_SEED = 42

In [3]:
COL_NOTES = {
    "id":           "Ad identifier (unique per row)",
    "click":        "Target: 1 = clicked, 0 = not clicked",
    "hour":         "YYMMDDHH format — encodes date AND hour",
    "C1":           "Anonymized categorical",
    "banner_pos":   "Position of the banner on the page",
    "site_id":      "Site identifier",
    "site_domain":  "Domain the site belongs to",
    "site_category":"Category of the site",
    "app_id":       "App identifier",
    "app_domain":   "Domain of the app",
    "app_category": "Category of the app",
    "device_id":    "Device identifier (very high cardinality)",
    "device_ip":    "IP address (very high cardinality)",
    "device_model": "Device model",
    "device_type":  "Type of device (phone/tablet/…)",
    "device_conn_type": "Connection type (wifi/3g/…)",
    "C14–C21":      "Anonymized categoricals",
}

for k,v in COL_NOTES.items():
    print(f"{k:20s}:   {v}")

id                  :   Ad identifier (unique per row)
click               :   Target: 1 = clicked, 0 = not clicked
hour                :   YYMMDDHH format — encodes date AND hour
C1                  :   Anonymized categorical
banner_pos          :   Position of the banner on the page
site_id             :   Site identifier
site_domain         :   Domain the site belongs to
site_category       :   Category of the site
app_id              :   App identifier
app_domain          :   Domain of the app
app_category        :   Category of the app
device_id           :   Device identifier (very high cardinality)
device_ip           :   IP address (very high cardinality)
device_model        :   Device model
device_type         :   Type of device (phone/tablet/…)
device_conn_type    :   Connection type (wifi/3g/…)
C14–C21             :   Anonymized categoricals


In [4]:
with gzip.open(TRAIN_PATH,'rb') as f:
    total_rows = sum(1 for _ in f)

print("Total number of rows:", total_rows)

Total number of rows: 40428968


In [5]:
skip_prob = 1-(SAMPLE_SIZE/total_rows)
skip_rows = np.random.RandomState(RANDOM_SEED).choice(
    range(1,total_rows+1),
    size = (total_rows-SAMPLE_SIZE),
    replace=False
)
print(skip_prob)

0.876326301477693


In [6]:
df = pd.read_csv(TRAIN_PATH,skiprows=skip_rows)
print(f"Loaded sample of rows {df.shape[0]} and columns {df.shape[1]}")
df.head()

Loaded sample of rows 5000000 and columns 24


,id,click,hour,C1,banner_pos,site_id,site_domain,site_category,app_id,app_domain,...,device_type,device_conn_type,C14,C15,C16,C17,C18,C19,C20,C21
0,10000679056417042096,0,14102100,1005,1,fe8cc448,9166c161,0569f928,ecad2386,7801e8d9,...,1,0,18993,320,50,2161,0,35,-1,157
1,10004482643316086592,0,14102100,1005,0,85f751fd,c4e18dd6,50e219e0,66a5f0f3,d9b5648e,...,1,0,21234,320,50,2434,3,163,100088,61
2,10004574413841529209,0,14102100,1005,0,1fbe01fe,f3845767,28905ebd,ecad2386,7801e8d9,...,1,0,15706,320,50,1722,0,35,-1,79
3,10007163879183388340,0,14102100,1005,0,030440fe,08ba7db9,76b2941d,ecad2386,7801e8d9,...,1,0,18993,320,50,2161,0,35,-1,157
4,10009147085943364421,0,14102100,1005,1,e151e245,7e091613,f028772b,ecad2386,7801e8d9,...,1,0,17037,320,50,1934,2,39,-1,16


In [7]:
df['hour_str'] = df['hour'].astype(str)
df['date'] = pd.to_datetime(df['hour_str'], format="%y%m%d%H")
df['hour_of_day'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['date_only'] = df['date'].dt.date
df['day'] = df['date'].dt.day

In [8]:
train = df[df['day']<=29]
val = df[df['day']==30]

In [9]:
y_train = train['click'].values
y_val = val['click'].values

In [10]:
BASE_FEATURES = [
    "C1", "banner_pos", "site_category", "app_category",
    "device_type", "device_conn_type", "C15", "C16", "C18",
    "hour_of_day", "day_of_week"
]

In [11]:
CAT_COLS = ["C1", "banner_pos", "site_category", "app_category",
            "device_type", "device_conn_type", "C15", "C16", "C18"]

In [12]:
def to_cat(df, cols):
    df = df.copy()
    for c in cols:
        df[c] = df[c].astype("category")
    return df
 
X_train_13 = to_cat(train[BASE_FEATURES], CAT_COLS)
X_val_13   = to_cat(val[BASE_FEATURES],   CAT_COLS)